In [5]:
import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, balanced_accuracy_score
from xgboost import XGBClassifier

In [6]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X = train.drop(columns=["id", "Irrigation_Need"])
y = train["Irrigation_Need"]

X_test = test.drop(columns=["id"])
test_ids = test["id"]

# one-hot encode for RF and XGBoost
X = pd.get_dummies(X, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# align columns
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [7]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scorer = make_scorer(balanced_accuracy_score)

In [8]:
sample_size = 100000 
train_sample = train.sample(sample_size, random_state=42)

X_sample = train_sample.drop(columns=["id", "Irrigation_Need"])
y_sample = train_sample["Irrigation_Need"]

X_sample = pd.get_dummies(X_sample, drop_first=True)
X_sample = X_sample.reindex(columns=X.columns, fill_value=0)

y_sample_encoded = le.transform(y_sample)


def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 400),
        "max_depth": trial.suggest_int("max_depth", 8, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestClassifier(**params)

    score = cross_val_score(
        model,
        X_sample,
        y_sample_encoded,
        cv=3,
        scoring="balanced_accuracy",
        n_jobs=-1
    ).mean()

    return score


study_rf = optuna.create_study(direction="maximize")

study_rf.optimize(objective_rf, n_trials=5)

print("Best RF score:", study_rf.best_value)
print("Best RF params:", study_rf.best_params)

[I 2026-04-22 21:53:06,288] A new study created in memory with name: no-name-b6af10e6-8baa-4889-bde2-0a90f34803db
[I 2026-04-22 21:53:31,464] Trial 0 finished with value: 0.9160744896613128 and parameters: {'n_estimators': 317, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.9160744896613128.
[I 2026-04-22 21:53:45,337] Trial 1 finished with value: 0.7384614948698371 and parameters: {'n_estimators': 335, 'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.9160744896613128.
[I 2026-04-22 21:54:08,551] Trial 2 finished with value: 0.9368040255264191 and parameters: {'n_estimators': 399, 'max_depth': 20, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.9368040255264191.
[I 2026-04-22 21:54:20,779] Trial 3 finished with value: 0.9159691963231399 and parameters: {'n_estimators': 239, 'max_depth': 12, 'm

Best RF score: 0.9368040255264191
Best RF params: {'n_estimators': 399, 'max_depth': 20, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt'}


In [9]:
best_rf = RandomForestClassifier(
    **study_rf.best_params,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X, y_encoded)

rf_preds = best_rf.predict(X_test)
rf_preds = le.inverse_transform(rf_preds)

bagging_submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": rf_preds
})

bagging_submission.to_csv("bagging_submission.csv", index=False)

In [10]:
sample_size = 100000 
train_sample = train.sample(sample_size, random_state=42)

X_sample = train_sample.drop(columns=["id", "Irrigation_Need"])
y_sample = train_sample["Irrigation_Need"]

X_sample = pd.get_dummies(X_sample, drop_first=True)
X_sample = X_sample.reindex(columns=X.columns, fill_value=0)

y_sample_encoded = le.transform(y_sample)


def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "objective": "multi:softprob",
        "num_class": 3,
        "random_state": 42,
        "eval_metric": "mlogloss",
        "n_jobs": -1,
        "tree_method": "hist"
    }

    model = XGBClassifier(**params)

    score = cross_val_score(
        model,
        X_sample,  
        y_sample_encoded,
        cv=3,        
        scoring="balanced_accuracy",
        n_jobs=1                 
    ).mean()

    return score


study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=5)

print("Best XGB score:", study_xgb.best_value)
print("Best XGB params:", study_xgb.best_params)

[I 2026-04-22 21:57:02,093] A new study created in memory with name: no-name-169f1f89-4088-4650-bc12-37d7f6fe1821
[I 2026-04-22 21:57:08,334] Trial 0 finished with value: 0.9570553176293711 and parameters: {'n_estimators': 318, 'max_depth': 8, 'learning_rate': 0.09389503825935587, 'subsample': 0.8256258044167195, 'colsample_bytree': 0.8239901427056938, 'min_child_weight': 3, 'gamma': 1.2173842420066008}. Best is trial 0 with value: 0.9570553176293711.
[I 2026-04-22 21:57:17,237] Trial 1 finished with value: 0.9568256208169311 and parameters: {'n_estimators': 323, 'max_depth': 5, 'learning_rate': 0.04949844559621916, 'subsample': 0.9402708432491074, 'colsample_bytree': 0.8484434960301037, 'min_child_weight': 3, 'gamma': 1.6536012095243828}. Best is trial 0 with value: 0.9570553176293711.
[I 2026-04-22 21:57:23,090] Trial 2 finished with value: 0.957430354161812 and parameters: {'n_estimators': 314, 'max_depth': 6, 'learning_rate': 0.1371649347563986, 'subsample': 0.9135667685669274, 'co

Best XGB score: 0.957430354161812
Best XGB params: {'n_estimators': 314, 'max_depth': 6, 'learning_rate': 0.1371649347563986, 'subsample': 0.9135667685669274, 'colsample_bytree': 0.8768712112683614, 'min_child_weight': 2, 'gamma': 1.068228874130451}


In [11]:
best_xgb = XGBClassifier(
    **study_xgb.best_params,
    objective="multi:softmax",
    num_class=3,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1
)

best_xgb.fit(X, y_encoded)

xgb_preds = best_xgb.predict(X_test)
xgb_preds = le.inverse_transform(xgb_preds)

boosting_submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": xgb_preds
})

boosting_submission.to_csv("boosting_submission.csv", index=False)

## Week 4: 

In [21]:
import catboost as cb
import lightgbm as lgb

## CatBoost: Configuration Comparison

In [29]:
# CatBoost Configuration 1: Conservative (fewer iterations, shallow depth, high learning rate)
cb_config1 = cb.CatBoostClassifier(
    iterations=100,
    depth=3,
    learning_rate=0.1,
    bootstrap_type='Bernoulli',
    subsample=0.8,
    min_data_in_leaf=5,
    random_state=42,
    verbose=False
)

cb_config1.fit(X, y_encoded)
cb_config1_preds = cb_config1.predict(X_test)
cb_config1_preds = le.inverse_transform(cb_config1_preds)

# Cross-validation score for comparison
cb_config1_score = cross_val_score(
    cb_config1,
    X,
    y_encoded,
    cv=cv,
    scoring="balanced_accuracy"
).mean()

print("CatBoost Config 1:")
print(f"  CV Balanced Accuracy: {cb_config1_score:.4f}")
print(f"  Parameters: iterations=100, depth=3, learning_rate=0.1, subsample=0.8")


/Users/kellyohata/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


CatBoost Config 1:
  CV Balanced Accuracy: 0.9496
  Parameters: iterations=100, depth=3, learning_rate=0.1, subsample=0.8


In [30]:
# CatBoost Configuration 2: Moderate (balanced approach)
cb_config2 = cb.CatBoostClassifier(
    iterations=300,
    depth=5,
    learning_rate=0.05,
    bootstrap_type='Bernoulli',
    subsample=0.9,
    min_data_in_leaf=2,
    random_state=42,
    verbose=False
)

cb_config2.fit(X, y_encoded)
cb_config2_preds = cb_config2.predict(X_test)
cb_config2_preds = le.inverse_transform(cb_config2_preds)

cb_config2_score = cross_val_score(
    cb_config2,
    X,
    y_encoded,
    cv=cv,
    scoring="balanced_accuracy"
).mean()

print("CatBoost Config 2:")
print(f"  CV Balanced Accuracy: {cb_config2_score:.4f}")
print(f"  Parameters: iterations=300, depth=5, learning_rate=0.05, subsample=0.9")


/Users/kellyohata/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


CatBoost Config 2:
  CV Balanced Accuracy: 0.9601
  Parameters: iterations=300, depth=5, learning_rate=0.05, subsample=0.9


In [31]:
# CatBoost Configuration 3: Aggressive (more iterations, deeper trees, aggressive learning)
cb_config3 = cb.CatBoostClassifier(
    iterations=500,
    depth=7,
    learning_rate=0.15,
    bootstrap_type='Bernoulli',
    subsample=0.95,
    min_data_in_leaf=1,
    random_state=42,
    verbose=False
)

cb_config3.fit(X, y_encoded)
cb_config3_preds = cb_config3.predict(X_test)
cb_config3_preds = le.inverse_transform(cb_config3_preds)

cb_config3_score = cross_val_score(
    cb_config3,
    X,
    y_encoded,
    cv=cv,
    scoring="balanced_accuracy"
).mean()

print("CatBoost Config 3:")
print(f"  CV Balanced Accuracy: {cb_config3_score:.4f}")
print(f"  Parameters: iterations=500, depth=7, learning_rate=0.15, subsample=0.95")

print("\nCatBoost Summary:")
print(f"  Config 1: {cb_config1_score:.4f}")
print(f"  Config 2: {cb_config2_score:.4f}")
print(f"  Config 3: {cb_config3_score:.4f}")

# Choose best CatBoost model for submission
cb_configs = [
    (cb_config1_preds, cb_config1_score, "CB_Config1"),
    (cb_config2_preds, cb_config2_score, "CB_Config2"),
    (cb_config3_preds, cb_config3_score, "CB_Config3")
]

best_cb_preds, best_cb_score, best_cb_name = max(cb_configs, key=lambda x: x[1])

cb_submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": best_cb_preds
})

cb_submission.to_csv(f"cb_{best_cb_name}.csv", index=False)
print(f"\nBest CatBoost: {best_cb_name} with score {best_cb_score:.4f}")


/Users/kellyohata/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


CatBoost Config 3:
  CV Balanced Accuracy: 0.9613
  Parameters: iterations=500, depth=7, learning_rate=0.15, subsample=0.95

CatBoost Summary:
  Config 1: 0.9496
  Config 2: 0.9601
  Config 3: 0.9613

Best CatBoost: CB_Config3 with score 0.9613


## LightGBM: Configuration Comparison

In [ ]:
# LightGBM Configuration 1:  (fewer leaves)
lgb_config1 = lgb.LGBMClassifier(
    num_leaves=31,
    learning_rate=0.1,
    n_estimators=100,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

lgb_config1.fit(X, y_encoded)
lgb_config1_preds = lgb_config1.predict(X_test)
lgb_config1_preds = le.inverse_transform(lgb_config1_preds)

lgb_config1_score = cross_val_score(
    lgb_config1,
    X,
    y_encoded,
    cv=cv,
    scoring="balanced_accuracy"
).mean()

print("LightGBM Config 1:")
print(f"  CV Balanced Accuracy: {lgb_config1_score:.4f}")
print(f"  Parameters: num_leaves=31, learning_rate=0.1, n_estimators=100, min_child_samples=20")


LightGBM Config 1:
  CV Balanced Accuracy: 0.9615
  Parameters: num_leaves=31, learning_rate=0.1, n_estimators=100, min_child_samples=20


In [ ]:
# LightGBM Configuration 2: 
lgb_config2 = lgb.LGBMClassifier(
    num_leaves=63,
    learning_rate=0.05,
    n_estimators=300,
    min_child_samples=10,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    verbose=-1
)

lgb_config2.fit(X, y_encoded)
lgb_config2_preds = lgb_config2.predict(X_test)
lgb_config2_preds = le.inverse_transform(lgb_config2_preds)

lgb_config2_score = cross_val_score(
    lgb_config2,
    X,
    y_encoded,
    cv=cv,
    scoring="balanced_accuracy"
).mean()

print("LightGBM Config 2:")
print(f"  CV Balanced Accuracy: {lgb_config2_score:.4f}")
print(f"  Parameters: num_leaves=63, learning_rate=0.05, n_estimators=300, min_child_samples=10")


LightGBM Config 2:
  CV Balanced Accuracy: 0.9616
  Parameters: num_leaves=63, learning_rate=0.05, n_estimators=300, min_child_samples=10


In [ ]:
# LightGBM Configuration 3:  (more leaves, less regularization)
lgb_config3 = lgb.LGBMClassifier(
    num_leaves=127,
    learning_rate=0.15,
    n_estimators=500,
    min_child_samples=5,
    subsample=0.95,
    colsample_bytree=0.95,
    random_state=42,
    verbose=-1
)

lgb_config3.fit(X, y_encoded)
lgb_config3_preds = lgb_config3.predict(X_test)
lgb_config3_preds = le.inverse_transform(lgb_config3_preds)

lgb_config3_score = cross_val_score(
    lgb_config3,
    X,
    y_encoded,
    cv=cv,
    scoring="balanced_accuracy"
).mean()

print("LightGBM Config 3:")
print(f"  CV Balanced Accuracy: {lgb_config3_score:.4f}")
print(f"  Parameters: num_leaves=127, learning_rate=0.15, n_estimators=500, min_child_samples=5")

print("\nLightGBM Summary:")
print(f"  Config 1: {lgb_config1_score:.4f}")
print(f"  Config 2: {lgb_config2_score:.4f}")
print(f"  Config 3: {lgb_config3_score:.4f}")

# Choose best LightGBM model for submission
lgb_configs = [
    (lgb_config1_preds, lgb_config1_score, "LGB_Config1"),
    (lgb_config2_preds, lgb_config2_score, "LGB_Config2"),
    (lgb_config3_preds, lgb_config3_score, "LGB_Config3")
]

best_lgb_preds, best_lgb_score, best_lgb_name = max(lgb_configs, key=lambda x: x[1])

lgb_submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": best_lgb_preds
})

lgb_submission.to_csv(f"lgb_{best_lgb_name}.csv", index=False)
print(f"\nBest LightGBM: {best_lgb_name} with score {best_lgb_score:.4f}")


LightGBM Config 3:
  CV Balanced Accuracy: 0.9600
  Parameters: num_leaves=127, learning_rate=0.15, n_estimators=500, min_child_samples=5

LightGBM Summary:
  Config 1: 0.9615
  Config 2: 0.9616
  Config 3: 0.9600

Best LightGBM: LGB_Config2 with score 0.9616


## Model Comparison & Ensemble

In [35]:
# Comprehensive model comparison
print("\nBOOSTING MODEL COMPARISON")
print()

all_models = [
    ("XGBoost (Optuna-tuned)", study_xgb.best_value),
    ("CatBoost Config 1", cb_config1_score),
    ("CatBoost Config 2", cb_config2_score),
    ("CatBoost Config 3", cb_config3_score),
    ("LightGBM Config 1", lgb_config1_score),
    ("LightGBM Config 2", lgb_config2_score),
    ("LightGBM Config 3", lgb_config3_score),
]

# Sort by score
sorted_models = sorted(all_models, key=lambda x: x[1], reverse=True)

print(f"\n{'Model':<40} {'CV Balanced Accuracy':<20}")
for i, (name, score) in enumerate(sorted_models, 1):
    print(f"{i}. {name:<38} {score:.4f}")


BOOSTING MODEL COMPARISON


Model                                    CV Balanced Accuracy
1. LightGBM Config 2                      0.9616
2. LightGBM Config 1                      0.9615
3. CatBoost Config 3                      0.9613
4. CatBoost Config 2                      0.9601
5. LightGBM Config 3                      0.9600
6. XGBoost (Optuna-tuned)                 0.9574
7. CatBoost Config 1                      0.9496
